In [0]:
"""
Data Quality - Meaning Validity Check Task
Validates data format/meaning (SIREN, SIRET, email, phone, VAT, etc.)
"""
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
import json
import re

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("column_types", "{}")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
COLUMN_TYPES = json.loads(dbutils.widgets.get("column_types"))

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Meaning Validity Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Column types to validate: {COLUMN_TYPES}")

# ==================================================
# HELPERS (column-expr friendly)
# ==================================================
def _null_or_empty(c):
    """Check if column is null or empty after trimming"""
    return c.isNull() | (F.length(F.trim(c.cast("string"))) == 0)

def _digits_only(c):
    """Extract only digits from column"""
    return F.regexp_replace(c.cast("string"), r"\D", "")

# ==================================================
# Luhn algorithm UDF
# ==================================================
@F.udf(BooleanType())
def luhn_check(s):
    """Validate using Luhn algorithm"""
    if s is None:
        return None
    ss = re.sub(r"\D", "", str(s))
    if not ss:
        return None
    total = 0
    alt = False
    for ch in ss[::-1]:
        d = ord(ch) - 48
        if d < 0 or d > 9:
            return False
        if alt:
            d *= 2
            if d > 9:
                d -= 9
        total += d
        alt = not alt
    return (total % 10) == 0

# ==================================================
# Convert boolean column to 0/1/None integer
# ==================================================
def as_010(col_bool):
    """Convert boolean column to integer: None → None, True → 1, False → 0"""
    return (F.when(col_bool.isNull(), F.lit(None).cast("int"))
             .when(col_bool, F.lit(1))
             .otherwise(F.lit(0)))

# ==================================================
# Validators → return Column of IntegerType (None/0/1)
# ==================================================
def is_valid_siren(c):
    """SIREN: 9 digits + Luhn (with auto-padding for missing leading zeros)"""
    d = F.lpad(_digits_only(c), 9, "0")
    cond = (F.length(d) == 9) & c.isNotNull()
    return as_010(F.when(_null_or_empty(c), F.lit(None))
                   .otherwise(F.when(cond & luhn_check(d), F.lit(True)).otherwise(F.lit(False))))

def is_valid_siret(c):
    """SIRET: 14 digits + Luhn (with auto-padding for missing leading zeros)"""
    d = F.lpad(_digits_only(c), 14, "0")
    cond = (F.length(d) == 14) & c.isNotNull()
    return as_010(F.when(_null_or_empty(c), F.lit(None))
                   .otherwise(F.when(cond & luhn_check(d), F.lit(True)).otherwise(F.lit(False))))

def is_valid_french_vat(c):
    """French VAT: FR + 2 alphanumeric + 9 digits (SIREN part must pass Luhn)"""
    # Normalize: remove spaces/dots, convert to uppercase
    s = F.upper(F.regexp_replace(c.cast("string"), r"[.\s]", ""))
    
    # Extract the 9-digit SIREN part (characters 5-13, i.e., positions after "FRXX")
    siren_part = F.substring(s, 5, 9)
    
    # Format check: FR + 2 alphanumeric + 9 digits
    format_ok = s.rlike(r"^FR[A-Z0-9]{2}\d{9}$")
    
    # Luhn check on the 9-digit SIREN part
    luhn_ok = luhn_check(_digits_only(siren_part))
    
    # Combined condition
    cond = format_ok & luhn_ok
    
    return as_010(F.when(_null_or_empty(c), F.lit(None))
                   .when(s == F.lit("NONASSUJETTI"), F.lit(None))
                   .otherwise(cond))

def is_valid_email(c):
    """Email validation"""
    s = c.cast("string")
    regex = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_phone(c):
    """Phone number validation"""
    s = c.cast("string")
    regex = r"^\+?[\d\s\-\(\)]{10,}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_postal_code(c):
    """Postal code (5 digits)"""
    s = F.trim(c.cast("string"))
    regex = r"^\d{5}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_url(c):
    """URL validation"""
    s = c.cast("string")
    regex = r"^https?://[^\s]+$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_ipv4(c):
    """IPv4 address validation"""
    s = F.trim(c.cast("string"))
    regex = r"^(\d{1,3}\.){3}\d{1,3}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_date_yyyy_mm(c):
    """Date format: YYYY-MM"""
    s = F.trim(c.cast("string"))
    regex = r"^\d{4}-\d{2}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_date_dd_mm_yyyy(c):
    """Date format: DD/MM/YYYY"""
    s = F.trim(c.cast("string"))
    regex = r"^\d{2}/\d{2}/\d{4}$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_integer(c):
    """Integer: optional leading/trailing whitespace, optional sign, one or more digits.
    Null/empty → None (not scored). Non-numeric string → 0 (invalid)."""
    s = F.trim(c.cast("string"))
    regex = r"^-?\d+$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_decimal(c):
    """Decimal: integer or decimal number, dot or comma as separator.
    Null/empty → None (not scored). Non-numeric string → 0 (invalid)."""
    s = F.trim(c.cast("string"))
    # Accepts: -123  123.45  123,45  .5  ,5
    regex = r"^-?\d*[.,]?\d+$"
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(s.rlike(regex)))

def is_valid_string(c):
    """String: any non-null, non-empty value is valid.
    Null/empty → None (not scored). Non-empty string → 1 (valid)."""
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(F.lit(True)))

def is_valid_boolean(c):
    """Boolean: accepts true/false/yes/no/1/0 (case-insensitive).
    Null/empty → None (not scored). Any other value → 0 (invalid)."""
    s = F.lower(F.trim(c.cast("string")))
    valid_values = ["true", "false", "yes", "no", "1", "0"]
    cond = s.isin(valid_values)
    return as_010(F.when(_null_or_empty(c), F.lit(None)).otherwise(cond))

# ==================================================
# FORMAT → VALIDATOR router
# ==================================================
def validator_for(fmt_str, col_expr):
    """Route format string to appropriate validator function"""
    fmt = (fmt_str or "").strip().lower()
    
    if fmt == "siren":                  return is_valid_siren(col_expr)
    if fmt == "siret":                  return is_valid_siret(col_expr)
    if fmt == "french vat":             return is_valid_french_vat(col_expr)
    if fmt == "email":                  return is_valid_email(col_expr)
    if fmt == "phone":                  return is_valid_phone(col_expr)
    if fmt == "postal_code":            return is_valid_postal_code(col_expr)
    if fmt == "url":                    return is_valid_url(col_expr)
    if fmt == "ipv4":                   return is_valid_ipv4(col_expr)
    if fmt == "date yyyy-mm":           return is_valid_date_yyyy_mm(col_expr)
    if fmt == "date dd/mm/yyyy":        return is_valid_date_dd_mm_yyyy(col_expr)
    if fmt == "integer":                return is_valid_integer(col_expr)
    if fmt == "decimal":                return is_valid_decimal(col_expr)
    if fmt == "string":                 return is_valid_string(col_expr)
    if fmt == "boolean":                return is_valid_boolean(col_expr)
    
    # Unknown format → return None
    return F.lit(None).cast("int")

# ==================================================
# Read input data
# ==================================================
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

# Start with row_id only
df_checks = df_input.select("row_id")

# Output table name - CHANGED to validity (not meaning_validity)
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_validity"

# Check if validity table already exists (from comparison_validity_check)
try:
    df_existing = spark.table(output_table)
    print(f"Found existing validity table with {df_existing.count()} rows")
    existing_columns = set(df_existing.columns) - {'row_id', 'execution_timestamp'}
    print(f"Existing check columns: {sorted(existing_columns)}")
    df_checks = df_existing.drop("execution_timestamp")  # Start with existing data
except:
    print("No existing validity table found, creating new one")
    df_checks = df_input.select("row_id")
    existing_columns = set()

# ==================================================
# Apply validations for each column
# ==================================================
if not COLUMN_TYPES:
    print("No column types specified for meaning validity check")
else:
    print(f"\nCreating meaning validity checks for {len(COLUMN_TYPES)} columns:")
    
    for col_name, data_type in COLUMN_TYPES.items():
        check_col_name = f"{col_name}_validity_check"
        
        print(f"  Processing column: {col_name} (type: {data_type}) → {check_col_name}")
        
        if col_name not in df_input.columns:
            print(f"    Warning: Column '{col_name}' not found in input, skipping")
            continue
        
        # Get validator expression for this data type
        expr = validator_for(data_type, F.col(col_name))
        
        # Create new check column
        df_new_check = df_input.select("row_id", col_name).withColumn(check_col_name, expr).select("row_id", check_col_name)
        
        # If this column already has a validity check, merge them
        if check_col_name in existing_columns:
            print(f"    Merging with existing validity check for {col_name}")
            # Merge logic: 0 if ANY is 0, 1 if ALL are 1, None if both are None
            df_checks = df_checks.join(
                df_new_check.withColumnRenamed(check_col_name, f"{check_col_name}_new"),
                "row_id",
                "left"
            )
            
            df_checks = df_checks.withColumn(
                check_col_name,
                F.when(
                    (F.col(check_col_name) == 0) | (F.col(f"{check_col_name}_new") == 0), 0  # ANY failure = fail
                ).when(
                    (F.col(check_col_name) == 1) & (F.col(f"{check_col_name}_new") == 1), 1  # ALL pass = pass
                ).when(
                    F.col(check_col_name).isNull() & F.col(f"{check_col_name}_new").isNull(), None  # Both null = null
                ).when(
                    F.col(check_col_name).isNull(), F.col(f"{check_col_name}_new")  # One null, use other
                ).otherwise(F.col(check_col_name))  # One null, use other
            ).drop(f"{check_col_name}_new")
        else:
            # New column, just join it
            df_checks = df_checks.join(df_new_check, "row_id", "left")
        
        print(f"    Applied validation for type: {data_type}")

# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog
df_checks.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")

# Display statistics
print("\nMeaning Validity Statistics:")
for col_name, data_type in COLUMN_TYPES.items():
    check_col_name = f"{col_name}_validity_check"
    
    if check_col_name in df_checks.columns:
        valid_count = df_checks.filter(F.col(check_col_name) == 1).count()
        invalid_count = df_checks.filter(F.col(check_col_name) == 0).count()
        null_count = df_checks.filter(F.col(check_col_name).isNull()).count()
        
        # Calculate percentage based on non-null values only
        non_null_count = valid_count + invalid_count
        validity_pct = (valid_count / non_null_count * 100) if non_null_count > 0 else 0
        
        print(f"  {col_name} ({data_type}): {validity_pct:.2f}% valid ({valid_count}/{non_null_count} non-null, {null_count} null)")